# Stage 2 Exoskeleton Rollout Analysis

This notebook focuses on simple, robust rollout diagnostics for the frozen-walker Stage 2 bootstrap experiment.

What this notebook emphasizes:
- time-series plots for torque, effort, and joint motion
- autocorrelation to assess whether the torque signals are periodic
- left/right cross-correlation to assess whether the torques overlap or are phase-shifted
- summary metrics for effort, torque magnitude, and smoothness

What this notebook de-emphasizes:
- no gait-cycle peak detection
- no phase-scatter clouds as a main result
- no strong biomechanical claims from approximate internal phase proxies

Important limitations:
- this is a frozen-walker bootstrap experiment, not full human-exo co-adaptation
- `effort` is a proxy signal, not a physiological ground-truth outcome
- `phi_r` and `phi_l` reconstructed from sin/cos are approximate controller-state proxies, not true gait events


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

plt.style.use("default")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

rollout_name = "stage2_eval_rollout.npz"
rollout_path = None  # Optional override, e.g. Path("/absolute/path/to/stage2_eval_rollout.npz")

candidate_paths = [
    Path("rl_output") / rollout_name,
    Path("rl") / "rl_output" / rollout_name,
    Path("..") / "rl_output" / rollout_name,
    Path("..") / ".." / "rl_output" / rollout_name,
]

if rollout_path is not None:
    rollout_path = Path(rollout_path).expanduser().resolve()

if rollout_path is None:
    for candidate in candidate_paths:
        if candidate.exists():
            rollout_path = candidate.resolve()
            break

if rollout_path is None:
    matches = sorted(Path.cwd().rglob(rollout_name))
    if matches:
        rollout_path = matches[0].resolve()

if rollout_path is None:
    raise FileNotFoundError(
        "Could not find rl_output/stage2_eval_rollout.npz. Set rollout_path manually in this cell if needed."
    )

baseline_effort = None
baseline_path = None
for candidate in [
    rollout_path.parent / "exo_training_log.json",
    Path("rl_output/exo_training_log.json"),
    Path("rl/rl_output/exo_training_log.json"),
    Path("../rl_output/exo_training_log.json"),
]:
    if candidate.exists():
        baseline_path = candidate.resolve()
        with open(baseline_path) as f:
            baseline_effort = json.load(f).get("baseline_effort")
        break

rollout_path

## Section 1 — Load Data

In [ ]:
with np.load(rollout_path) as data:
    raw = {k: np.asarray(data[k]).reshape(-1) for k in data.files}

if "sample" not in raw and "step" in raw:
    raw["sample"] = raw["step"]

required = [
    "episode",
    "sample",
    "tau_r",
    "tau_l",
    "effort",
    "hip_r",
    "hip_l",
    "hipd_r",
    "hipd_l",
]
missing = [k for k in required if k not in raw]
if missing:
    raise KeyError(f"Missing required rollout keys: {missing}")

df = pd.DataFrame(raw)
df = df.sort_values(["episode", "sample"]).reset_index(drop=True)
df["sample_in_episode"] = df.groupby("episode").cumcount()

phase_note = None
if {"sin_phi_r", "cos_phi_r", "sin_phi_l", "cos_phi_l"}.issubset(df.columns):
    df["phi_r"] = np.arctan2(df["sin_phi_r"], df["cos_phi_r"])
    df["phi_l"] = np.arctan2(df["sin_phi_l"], df["cos_phi_l"])
elif {"sin_phi", "cos_phi"}.issubset(df.columns):
    df["phi_r"] = np.arctan2(df["sin_phi"], df["cos_phi"])
    df["phi_l"] = df["phi_r"]
    phase_note = "Only generic `sin_phi` / `cos_phi` were found, so the same phase proxy is used for both sides."
else:
    raise KeyError(
        "Missing phase proxy columns. Expected `sin_phi_r`, `cos_phi_r`, `sin_phi_l`, `cos_phi_l` or the older `sin_phi`, `cos_phi`."
    )

df["phi_r_pct"] = np.mod(df["phi_r"], 2 * np.pi) / (2 * np.pi) * 100
df["phi_l_pct"] = np.mod(df["phi_l"], 2 * np.pi) / (2 * np.pi) * 100

episode_lengths = (
    df.groupby("episode")
    .agg(n_samples=("sample_in_episode", "size"))
    .sort_index()
)

display(Markdown(f"- Rollout file: `{rollout_path}`"))
if baseline_effort is not None:
    display(Markdown(f"- Baseline effort loaded from `{baseline_path}`: `{baseline_effort:.6f}`"))
else:
    display(Markdown("- Baseline effort not found nearby. Effort reduction vs baseline may be undetermined."))
if phase_note is not None:
    display(Markdown(f"- {phase_note}"))

display(pd.DataFrame({"column": df.columns}))
display(episode_lengths)
df.head()

## Section 2 — Basic Time-Series Plots

The default view below uses the longest episode, which is usually the clearest episode for quick inspection. Change `selected_episode` if you want a different one.

In [ ]:
selected_episode = int(episode_lengths["n_samples"].idxmax())
ep = df.loc[df["episode"] == selected_episode].copy()

plot_groups = [
    ("Torques", [("tau_r", "tab:blue"), ("tau_l", "tab:orange")]),
    ("Effort", [("effort", "tab:green")]),
    ("Hip Angles", [("hip_r", "tab:blue"), ("hip_l", "tab:orange")]),
]
if "pelvis_vx" in ep.columns:
    plot_groups.append(("Pelvis Velocity", [("pelvis_vx", "tab:red")]))
if "torso_pitch" in ep.columns:
    plot_groups.append(("Torso Pitch", [("torso_pitch", "tab:brown")]))

fig, axes = plt.subplots(len(plot_groups), 1, figsize=(11, 2.6 * len(plot_groups)), sharex=True)
if len(plot_groups) == 1:
    axes = [axes]

for ax, (title, columns) in zip(axes, plot_groups):
    for name, color in columns:
        ax.plot(ep["sample_in_episode"], ep[name], lw=1.8, color=color, label=name)
    ax.set_title(f"Episode {selected_episode}: {title}")
    ax.legend(loc="upper right")
    ax.set_ylabel(title)

axes[-1].set_xlabel("sample within episode")
plt.tight_layout()
plt.show()

## Section 3 — Left/Right Coordination Analysis

This section uses only time-domain correlations.

- Autocorrelation of `tau_r` and `tau_l` is used as a practical periodicity check.
- Cross-correlation of `tau_r` and `tau_l` is used to estimate overlap or phase shift.
- Positive cross-correlation lag here means `tau_l` lags behind `tau_r` in the notebook's convention.


In [ ]:
def lagged_corr_curve(x, y, max_lag=None):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = min(len(x), len(y))
    x = x[:n]
    y = y[:n]

    if n < 4:
        return np.array([], dtype=int), np.array([], dtype=float)

    if max_lag is None:
        max_lag = n // 2
    max_lag = int(min(max_lag, n - 2))
    if max_lag < 1:
        return np.array([], dtype=int), np.array([], dtype=float)

    lags = np.arange(-max_lag, max_lag + 1)
    corr = np.full(len(lags), np.nan, dtype=float)

    for i, lag in enumerate(lags):
        if lag < 0:
            x_seg = x[-lag:]
            y_seg = y[: n + lag]
        elif lag > 0:
            x_seg = x[: n - lag]
            y_seg = y[lag:]
        else:
            x_seg = x
            y_seg = y

        if len(x_seg) < 3:
            continue

        x0 = x_seg - x_seg.mean()
        y0 = y_seg - y_seg.mean()
        denom = np.linalg.norm(x0) * np.linalg.norm(y0)
        corr[i] = np.dot(x0, y0) / denom if denom > 0 else np.nan

    return lags, corr

def periodicity_curve(x, min_lag=None, max_lag=None):
    x = np.asarray(x, dtype=float)
    n = len(x)
    if n < 4:
        return np.array([], dtype=int), np.array([], dtype=float), np.nan, np.nan

    if max_lag is None:
        max_lag = n // 2
    max_lag = int(min(max_lag, n - 2))
    if min_lag is None:
        min_lag = max(2, n // 20)
    min_lag = int(min(max(min_lag, 1), max_lag))

    lags, corr = lagged_corr_curve(x, x, max_lag=max_lag)
    if len(lags) == 0:
        return np.array([], dtype=int), np.array([], dtype=float), np.nan, np.nan

    keep = lags > 0
    lags = lags[keep]
    corr = corr[keep]
    search = lags >= min_lag

    if not np.any(search) or np.all(np.isnan(corr[search])):
        return lags, corr, np.nan, np.nan

    best_idx = int(np.nanargmax(corr[search]))
    best_lag = int(lags[search][best_idx])
    best_corr = float(corr[search][best_idx])
    return lags, corr, best_lag, best_corr

coord_rows = []
selected_curves = {}

for episode_id, group in df.groupby("episode", sort=True):
    tau_r = group["tau_r"].to_numpy()
    tau_l = group["tau_l"].to_numpy()

    ac_lags_r, ac_corr_r, tau_r_period, tau_r_periodicity = periodicity_curve(tau_r)
    ac_lags_l, ac_corr_l, tau_l_period, tau_l_periodicity = periodicity_curve(tau_l)

    cc_lags, cc_corr = lagged_corr_curve(tau_r, tau_l, max_lag=max(2, len(group) // 2))
    if len(cc_lags) == 0 or np.all(np.isnan(cc_corr)):
        lr_lag = np.nan
        lr_corr = np.nan
    else:
        best_idx = int(np.nanargmax(cc_corr))
        lr_lag = int(cc_lags[best_idx])
        lr_corr = float(cc_corr[best_idx])

    period_ref = np.nanmean([tau_r_period, tau_l_period])
    lr_lag_pct = np.nan if not np.isfinite(period_ref) or period_ref == 0 else 100 * lr_lag / period_ref

    coord_rows.append(
        {
            "episode": int(episode_id),
            "n_samples": int(len(group)),
            "tau_r_period_samples": tau_r_period,
            "tau_r_periodicity": tau_r_periodicity,
            "tau_l_period_samples": tau_l_period,
            "tau_l_periodicity": tau_l_periodicity,
            "lr_lag_samples": lr_lag,
            "lr_lag_pct_of_period": lr_lag_pct,
            "lr_max_corr": lr_corr,
        }
    )

    if int(episode_id) == selected_episode:
        selected_curves = {
            "ac_lags_r": ac_lags_r,
            "ac_corr_r": ac_corr_r,
            "tau_r_period": tau_r_period,
            "tau_r_periodicity": tau_r_periodicity,
            "ac_lags_l": ac_lags_l,
            "ac_corr_l": ac_corr_l,
            "tau_l_period": tau_l_period,
            "tau_l_periodicity": tau_l_periodicity,
            "cc_lags": cc_lags,
            "cc_corr": cc_corr,
            "lr_lag": lr_lag,
            "lr_corr": lr_corr,
        }

coordination_table = pd.DataFrame(coord_rows).set_index("episode")
display(coordination_table)

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=False)

axes[0].plot(selected_curves["ac_lags_r"], selected_curves["ac_corr_r"], color="tab:blue", lw=2, label="tau_r autocorr")
axes[0].plot(selected_curves["ac_lags_l"], selected_curves["ac_corr_l"], color="tab:orange", lw=2, label="tau_l autocorr")
if np.isfinite(selected_curves["tau_r_period"]):
    axes[0].axvline(selected_curves["tau_r_period"], color="tab:blue", alpha=0.35, linestyle="--")
if np.isfinite(selected_curves["tau_l_period"]):
    axes[0].axvline(selected_curves["tau_l_period"], color="tab:orange", alpha=0.35, linestyle="--")
axes[0].set_title(f"Episode {selected_episode}: autocorrelation-based periodicity")
axes[0].set_xlabel("lag (samples)")
axes[0].set_ylabel("correlation")
axes[0].legend()

axes[1].plot(selected_curves["cc_lags"], selected_curves["cc_corr"], color="black", lw=2)
if np.isfinite(selected_curves["lr_lag"]):
    axes[1].axvline(selected_curves["lr_lag"], color="tab:red", alpha=0.5, linestyle="--")
axes[1].set_title(f"Episode {selected_episode}: cross-correlation between tau_r and tau_l")
axes[1].set_xlabel("lag (samples)")
axes[1].set_ylabel("correlation")

plt.tight_layout()
plt.show()

selected_lag = selected_curves["lr_lag"]
selected_corr = selected_curves["lr_corr"]
if np.isfinite(selected_lag):
    print(f"Selected episode best lag: {selected_lag:+.0f} samples")
    print(f"Selected episode max correlation: {selected_corr:.3f}")
else:
    print("Selected episode cross-correlation lag is undetermined.")

## Section 4 — Smoothness and Magnitude Metrics

In [ ]:
episode_effort = df.groupby("episode")["effort"].mean()

delta_tau_r_parts = [np.diff(g["tau_r"].to_numpy()) for _, g in df.groupby("episode", sort=True) if len(g) > 1]
delta_tau_l_parts = [np.diff(g["tau_l"].to_numpy()) for _, g in df.groupby("episode", sort=True) if len(g) > 1]
delta_tau_r = np.concatenate(delta_tau_r_parts) if delta_tau_r_parts else np.array([np.nan])
delta_tau_l = np.concatenate(delta_tau_l_parts) if delta_tau_l_parts else np.array([np.nan])

mean_effort = float(df["effort"].mean())
std_effort = float(df["effort"].std(ddof=0))
mean_abs_tau_r = float(np.abs(df["tau_r"]).mean())
mean_abs_tau_l = float(np.abs(df["tau_l"]).mean())
mean_abs_dtau_r = float(np.nanmean(np.abs(delta_tau_r)))
mean_abs_dtau_l = float(np.nanmean(np.abs(delta_tau_l)))
rms_tau_r = float(np.sqrt(np.mean(df["tau_r"] ** 2)))
rms_tau_l = float(np.sqrt(np.mean(df["tau_l"] ** 2)))
effort_cv = float(std_effort / max(abs(mean_effort), 1e-9))
smoothness_ratio = float(np.nanmean([
    mean_abs_dtau_r / max(mean_abs_tau_r, 1e-9),
    mean_abs_dtau_l / max(mean_abs_tau_l, 1e-9),
]))

summary = pd.Series(
    {
        "mean effort": mean_effort,
        "std effort": std_effort,
        "effort CV": effort_cv,
        "mean episode effort": float(episode_effort.mean()),
        "std episode effort": float(episode_effort.std(ddof=0)),
        "mean |tau_r|": mean_abs_tau_r,
        "mean |tau_l|": mean_abs_tau_l,
        "mean |Δtau_r|": mean_abs_dtau_r,
        "mean |Δtau_l|": mean_abs_dtau_l,
        "RMS tau_r": rms_tau_r,
        "RMS tau_l": rms_tau_l,
        "smoothness ratio": smoothness_ratio,
    }
)

if baseline_effort is not None and baseline_effort > 0:
    summary.loc["baseline effort"] = float(baseline_effort)
    summary.loc["effort change vs baseline (%)"] = float(100 * (mean_effort - baseline_effort) / baseline_effort)
    summary.loc["effort reduction vs baseline (%)"] = float(100 * (baseline_effort - mean_effort) / baseline_effort)

display(summary.to_frame("value"))

## Section 5 — Optional Phase-Aligned Line Plots

These plots are secondary diagnostics only.

- The x-axis is the reconstructed controller phase proxy, not a true gait event.
- The lines preserve time order and are broken at phase wrap points.
- These plots can help show whether torque follows a repeatable internal phase variable, but they should not be treated as full gait-phase analysis.


In [ ]:
def phase_line_arrays(phase_pct, value):
    phase_pct = np.asarray(phase_pct, dtype=float)
    value = np.asarray(value, dtype=float)
    split_idx = np.where(np.abs(np.diff(phase_pct)) > 50)[0] + 1
    phase_plot = np.insert(phase_pct, split_idx, np.nan)
    value_plot = np.insert(value, split_idx, np.nan)
    return phase_plot, value_plot

phi_r_plot, tau_r_plot = phase_line_arrays(ep["phi_r_pct"], ep["tau_r"])
phi_l_plot, tau_l_plot = phase_line_arrays(ep["phi_l_pct"], ep["tau_l"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

axes[0].plot(phi_r_plot, tau_r_plot, color="tab:blue", lw=1.6)
axes[0].set_title(f"Episode {selected_episode}: tau_r vs right phase proxy")
axes[0].set_xlabel("right phase proxy (%)")
axes[0].set_ylabel("torque")
axes[0].set_xlim(0, 100)

axes[1].plot(phi_l_plot, tau_l_plot, color="tab:orange", lw=1.6)
axes[1].set_title(f"Episode {selected_episode}: tau_l vs left phase proxy")
axes[1].set_xlabel("left phase proxy (%)")
axes[1].set_xlim(0, 100)

plt.tight_layout()
plt.show()

## Section 6 — Final Interpretation

In [ ]:
periodicity_values = coordination_table[["tau_r_periodicity", "tau_l_periodicity"]].to_numpy().ravel()
periodicity_values = periodicity_values[np.isfinite(periodicity_values)]
median_periodicity = float(np.median(periodicity_values)) if len(periodicity_values) else np.nan

lag_pct_values = coordination_table["lr_lag_pct_of_period"].to_numpy(dtype=float)
lag_pct_values = lag_pct_values[np.isfinite(lag_pct_values)]
median_abs_lag_pct = float(np.median(np.abs(lag_pct_values))) if len(lag_pct_values) else np.nan

lr_corr_values = coordination_table["lr_max_corr"].to_numpy(dtype=float)
lr_corr_values = lr_corr_values[np.isfinite(lr_corr_values)]
median_lr_corr = float(np.median(lr_corr_values)) if len(lr_corr_values) else np.nan

if not np.isfinite(median_periodicity):
    periodic_text = "Undetermined from autocorrelation."
elif median_periodicity >= 0.60:
    periodic_text = f"Yes. The torque signals look clearly periodic in time-domain autocorrelation (median periodicity score {median_periodicity:.2f})."
elif median_periodicity >= 0.35:
    periodic_text = f"Partly. The torque signals show some repeated structure, but periodicity is only moderate (median periodicity score {median_periodicity:.2f})."
else:
    periodic_text = f"Weakly. The torque signals do not show a strong repeated autocorrelation peak (median periodicity score {median_periodicity:.2f})."

if not np.isfinite(median_abs_lag_pct):
    lr_text = "Undetermined from cross-correlation."
elif median_abs_lag_pct <= 10 and (not np.isfinite(median_lr_corr) or median_lr_corr >= 0.5):
    lr_text = f"Mostly overlapping / synchronized. The typical lag magnitude is about {median_abs_lag_pct:.1f}% of the dominant repeat time."
elif median_abs_lag_pct <= 40:
    lr_text = f"Phase-shifted. The typical lag magnitude is about {median_abs_lag_pct:.1f}% of the dominant repeat time."
else:
    lr_text = f"Strongly offset. The typical lag magnitude is about {median_abs_lag_pct:.1f}% of the dominant repeat time."

if baseline_effort is None or baseline_effort <= 0:
    effort_change_text = "Effort reduction versus a baseline cannot be determined from the rollout alone because no nearby baseline reference was found."
else:
    reduction_pct = 100 * (baseline_effort - mean_effort) / baseline_effort
    if reduction_pct > 5:
        effort_change_text = f"Mean effort is lower than the saved baseline by about {reduction_pct:.1f}%."
    elif reduction_pct < -5:
        effort_change_text = f"Mean effort is higher than the saved baseline by about {-reduction_pct:.1f}%."
    else:
        effort_change_text = f"Mean effort is roughly unchanged relative to the saved baseline ({reduction_pct:.1f}%)."

if effort_cv < 0.20:
    effort_stability_text = "Effort looks fairly stable over the rollout."
elif effort_cv < 0.35:
    effort_stability_text = "Effort looks moderately stable, with some variation."
else:
    effort_stability_text = "Effort varies substantially over the rollout."

if smoothness_ratio < 0.15:
    smoothness_text = "The controller looks smooth by the sample-to-sample torque-change metric."
elif smoothness_ratio < 0.35:
    smoothness_text = "The controller looks moderately smooth, with noticeable but limited torque jitter."
else:
    smoothness_text = "The controller looks relatively noisy by the sample-to-sample torque-change metric."

summary_md = "## Final Summary\n\n"
summary_md += f"- **Are `tau_r` and `tau_l` periodic?** {periodic_text}\n"
summary_md += f"- **Are left and right torques overlapping or phase-shifted?** {lr_text}\n"
summary_md += f"- **Did effort decrease and stay stable?** {effort_change_text} {effort_stability_text}\n"
summary_md += f"- **Is the controller smooth or noisy?** {smoothness_text}\n"
summary_md += "- **Limitations:** This remains a frozen-walker bootstrap experiment; `effort` is a proxy, not a physiological endpoint; and `phi_r` / `phi_l` are approximate internal phase proxies, not true gait events.\n"

display(Markdown(summary_md))